# Implementazione blocco residuale custom

In questo notebook vogliamo capire come funziona un blocco residuale e perché viene usato nelle reti profonde.

L’idea principale è semplice: invece di far apprendere alla rete solo una trasformazione F(x), vogliamo che il blocco mantenga anche l’informazione originale tramite una *skip connection*.

La formula è:

$$y = F(x) + x$$

Così il gradiente può propagarsi più facilmente durante l’addestramento e la rete può imparare meglio anche quando ha molti layer.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers

def residual_block(x, filters, stride=1):
    """
    Blocco residuale con:
    -Conv 3x3 -> BN -> ReLU
    -Conv 3x3 -> BN
    -Skip Connection (con proiezione 1x1 se necessario)
    -ReLU finale dopo la somma

    Parametri 
    ----------
    x       : tensore di input
    filters : numero di filtri per entrambe le conv
    stride  : stride della prima conv (>1 ridue la risoluzione spaziale) 
    """

    shortcut = x

    # --- Ramo principale ---

    # Prima convoluzione
    x = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Seconda convoluzione 
    x = layers.Conv2D(filters, 3, strides=1, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)

    # --- Skip connection ---

    # Se stride != 1 oppure il numero di canali cambia non è possibile effettuare la somma

    #Soluzione: effettuiamo una convoluzione 1x1 per adattare alle dimensioni

    shortcut_channels = shortcut.shape[-1]

    if stride != 1 or shortcut_channels != filters:
        shortcut = layers.Conv2D(filters, kernel_size=1, strides=stride, use_bias=False, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    
    # Somma e attivazione finale 

    x = layers.Add()([x, shortcut])
    x = layers.ReLU()(x)
    return x


# Costruzione del modello CNN con blocchi residuali

Ora che abbiamo implementato il blocco residuale, possiamo costruire un'architettura completa sfruttandolo.

## Struttura dell'architettura

L'architettura segue questa pipeline:

Input (H, W, 3)

↓

Stem: Conv2D → BN → ReLU [estrae features di base]

↓

Stage 1: Residual Block → MaxPooling [riduce risoluzione, mantiene semantica]

↓

Stage 2: Residual Block (più filtri) → MaxPooling [aumenta feature complexity]

↓

Stage 3: Residual Block (ancora più filtri) → MaxPooling [rappresentazione 
astratta]

↓

Flatten [trasforma in vettore 1D]

↓

MLP: Dense → Dropout → ... → Output Softmax [classificazione finale]


## Strategia di aumento dei canali

- **Stage 1**: `FILTERS_STAGE_1` canali (es. 64)
- **Stage 2**: `FILTERS_STAGE_2` canali (es. 128) — raddoppiamo quando dimezziamo la risoluzione spaziale
- **Stage 3**: `FILTERS_STAGE_3` canali (es. 256) — compensiamo ulteriore perdita spaziale con più feature channels

Questo compromesso (meno spazio, più profondità dei canali) è la chiave delle ResNet moderne.





In [ ]:
import tensorflow as tf
from tensorflow.keras import Input, Model, layers

# Queste variabili hardcoded andrebbero spostate in un un file config.py

FILTERS_STAGE_1 = 64
FILTERS_STAGE_2 = 128
FILTERS_STAGE_3 = 256

MLP_UNITS = [256, 128]
DROPOUT_RATE = 0.4

KERNEL_SIZE = 3

IMG_SIZE = (128,128)
CHANNELS = 3

def build_model(num_classes, input_shape=None):
    """
    Costruisce la CNN residuale custom.

    Parametri
    ----------
    num_classes : int - numero di classi in output (30 per AID, 21 per UC)
    input_shape : tuple - (H, W, C), default da config
    """

    if input_shape is None:
        input_shape = (IMG_SIZE, CHANNELS)
    
    inputs = Input(shape=input_shape)

    # --- Stem: primo layer convoluzionale ---
    # Serve ad estrarre feature di base (bordi, texture) prima di entrare nei blocchi residuali

    x = layers.Conv2D(FILTERS_STAGE_1, kernel_size=KERNEL_SIZE, strides=1, padding='same',
                       use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # --- Stage 1: primo blocco residuale ---

    x = residual_block(x, filters=FILTERS_STAGE_1)
    x = layers.MaxPooling2D(pool_size=2)(x)

    # --- Stage 2: secondo blocco residuale ---

    x = residual_block(x, filters=FILTERS_STAGE_2)
    x = layers.MaxPooling2D(pool_size=2)(x)

    # --- Stage 3: terzo blocco residuale ---

    x = residual_block(x, filters=FILTERS_STAGE_3)
    x = layers.MaxPooling2D(pool_size=2)(x)

    # --- Flatten ---
    # Trasforma (H', W', C) in un vettore 1D.

    x = layers.Flatten()(x)

    # --- MLP ---

    for units in MLP_UNITS:
        x = layers.Dense(units, activation='relu')(x)
        x = layers.Dropout(DROPOUT_RATE)(x)
    
    # Output:  softmax su num_classes neuroni.
    # Produce una distribuzione di probabilità su tutte le classi 
    # Poichè abbiamo scelto loss categorical entropy è necessaria la softmax

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=inputs, outputs=outputs, name="ResidualCNN_custom")

    return model